# Flight Delay Volume Forecasting

## Project Overview

Forecasts **daily count of delayed flights** over a 14-day horizon. Synthetic data spans 2 years with weekly patterns, seasonal weather effects, and holiday congestion.

**Models**: Naive, LazyPredict, FLAML, StatsForecast. **Horizon**: 14 days.

## Learning Objectives

1. Understand time-series patterns (trend, seasonality, noise)
2. Build naive and seasonal-naive baselines
3. Engineer lag and rolling features for tabular ML
4. Use LazyPredict for quick regression benchmarking
5. Apply FLAML AutoML (excluding XGBoost due to compatibility)
6. Use StatsForecast (AutoARIMA, AutoETS, SeasonalNaive)
7. Compare all approaches with MAE / RMSE / MAPE

## Problem Statement

Given historical daily delayed flight counts, predict the next 14 days. Airlines and airports need delay forecasts for staffing, rebooking logistics, and proactive passenger communication.

## Why This Project Matters

Flight delays cost the US airline industry billions annually. Proactive delay prediction enables airlines to pre-position staff, prepare alternative routing, and communicate with passengers before disruptions cascade across the network.

## Dataset Overview

Synthetic dataset:
- 730 days (2 years) of daily delayed flight counts
- Weekly pattern (higher on busy travel days)
- Seasonal effects (winter storms, summer thunderstorms)
- Holiday congestion spikes
- Random weather disruption events

| Column | Description |
|--------|-------------|
| `date` | Date |
| `delayed_flights` | Daily count of delayed flights |

## Dataset Source and License Notes

Synthetically generated for educational purposes. No external download required.

## Environment Setup

In [1]:
import subprocess, importlib
for pkg in ['numpy', 'pandas', 'matplotlib', 'scikit-learn', 'statsforecast', 'flaml', 'lazypredict']:
    try:
        importlib.import_module(pkg.replace('-', '_').split('[')[0])
    except ImportError:
        subprocess.check_call(['pip', 'install', '-q', pkg])
print("All packages ready.")

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## Imports

In [2]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from sklearn.metrics import mean_absolute_error, mean_squared_error
print("Imports complete.")

Imports complete.


## Configuration / Constants

In [3]:
SEED = 42
N_POINTS = 730
HORIZON = 14
VAL_SIZE = 14
TEST_SIZE = 14
SEASON_LENGTH = 7
FREQ = 'D'
TARGET = 'delayed_flights'
np.random.seed(SEED)
print(f"Config: {N_POINTS} points, horizon={HORIZON}, season={SEASON_LENGTH}")

Config: 730 points, horizon=14, season=7


## Dataset Generation

In [4]:
dates = pd.date_range(start='2022-01-01', periods=N_POINTS, freq='D')
t = np.arange(N_POINTS)

base = 400 + 0.05 * t
weekly = np.zeros(N_POINTS)
for i in range(N_POINTS):
    dow = dates[i].dayofweek
    if dow == 4: weekly[i] = 80  # Friday congestion
    elif dow == 0: weekly[i] = 60  # Monday
    elif dow <= 3: weekly[i] = 20
    elif dow == 5: weekly[i] = -40  # Saturday fewer flights
    else: weekly[i] = -20

# Winter storms + summer thunderstorms (double peak)
seasonal = 60 * np.cos(2 * np.pi * (t - 15) / 365) + 40 * np.sin(2 * np.pi * (t - 180) / 182.5)

# Holiday congestion
holiday = np.zeros(N_POINTS)
for i in range(N_POINTS):
    m, d = dates[i].month, dates[i].day
    if m == 11 and 22 <= d <= 28: holiday[i] = 100
    elif (m == 12 and d >= 20) or (m == 1 and d <= 3): holiday[i] = 120
    elif m == 7 and 1 <= d <= 5: holiday[i] = 60

# Weather disruption events
disruptions = np.where(np.random.random(N_POINTS) < 0.08, np.random.uniform(100, 300, N_POINTS), 0)
noise = np.random.normal(0, 30, N_POINTS)

delayed_flights = base + weekly + seasonal + holiday + disruptions + noise
delayed_flights = np.maximum(delayed_flights, 50).round(0).astype(int)

df = pd.DataFrame({'date': dates, 'delayed_flights': delayed_flights})
print(f"Dataset shape: {df.shape}")
df.head()

Dataset shape: (730, 2)


,date,delayed_flights
0,2022-01-01,590
1,2022-01-02,529
2,2022-01-03,654
3,2022-01-04,464
4,2022-01-05,486


## Data Validation Checks

In [5]:
assert df.isnull().sum().sum() == 0, "Missing values"
assert (df[TARGET] > 0).all(), "Non-positive values"
assert df['date'].is_monotonic_increasing, "Not sorted"
assert len(df) == N_POINTS, "Row count mismatch"
print("All validation checks passed.")

All validation checks passed.


## Exploratory Data Analysis

In [6]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes[0, 0].plot(df['date'], df[TARGET], linewidth=0.6)
axes[0, 0].set_title('delayed_flights Over Time')
axes[0, 1].hist(df[TARGET], bins=30, edgecolor='black', alpha=0.7)
axes[0, 1].set_title('Distribution')
from pandas.plotting import autocorrelation_plot
autocorrelation_plot(df[TARGET], ax=axes[1, 0])
axes[1, 0].set_xlim(0, min(60, N_POINTS // 2))
axes[1, 0].set_title('Autocorrelation')
rw = min(SEASON_LENGTH * 2, N_POINTS // 4)
axes[1, 1].plot(df['date'], df[TARGET].rolling(rw).mean())
axes[1, 1].set_title(f'Rolling {rw}-period Mean')
plt.tight_layout()
plt.savefig('eda.png', dpi=100, bbox_inches='tight')
plt.show()
print("EDA complete.")

EDA complete.


## Target Analysis

In [7]:
print("delayed_flights Statistics:")
print(df[TARGET].describe())
print(f"\nCV: {df[TARGET].std() / df[TARGET].mean():.3f}")
print(f"Skewness: {df[TARGET].skew():.3f}")

delayed_flights Statistics:
count    730.000000
mean     464.131507
std       98.467652
min      255.000000
25%      395.000000
50%      448.000000
75%      517.000000
max      908.000000
Name: delayed_flights, dtype: float64

CV: 0.212
Skewness: 1.063


## Train / Validation / Test Split Strategy

Time-aware split (no shuffling):
- **Train**: all except last 28
- **Validation**: 14 points
- **Test**: last 14 points

In [8]:
train = df.iloc[:-(VAL_SIZE + TEST_SIZE)].copy()
val = df.iloc[-(VAL_SIZE + TEST_SIZE):-TEST_SIZE].copy()
test = df.iloc[-TEST_SIZE:].copy()
print(f"Train: {len(train)}, Val: {len(val)}, Test: {len(test)}")

Train: 702, Val: 14, Test: 14


## Preprocessing Strategy

Minimal preprocessing — tree models handle raw features. Features created next.

In [9]:
df_full = df.copy().sort_values('date').reset_index(drop=True)
print('Data ready.')

Data ready.


## Feature Engineering

In [10]:
def create_features(data):
    d = data.copy()
    d['dow'] = d['date'].dt.dayofweek
    d['month'] = d['date'].dt.month
    d['day'] = d['date'].dt.day
    d['week_of_year'] = d['date'].dt.isocalendar().week.astype(int)
    for lag in [1, 7, 14, 28]:
        d[f'lag_{lag}'] = d[TARGET].shift(lag)
    for w in [7, 14, 28]:
        d[f'rmean_{w}'] = d[TARGET].shift(1).rolling(w).mean()
        d[f'rstd_{w}'] = d[TARGET].shift(1).rolling(w).std()
    return d

df_feat = create_features(df_full).dropna().reset_index(drop=True)
feature_cols = [c for c in df_feat.columns if c not in ['date', TARGET]]
print(f"Features: {len(feature_cols)} columns, {len(df_feat)} rows")

Features: 14 columns, 702 rows


## Baseline Model — Naive Forecasts

In [11]:
def calc_metrics(y_true, y_pred, name=""):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / np.maximum(y_true, 1))) * 100
    print(f"{name:30s} | MAE: {mae:10.1f} | RMSE: {rmse:10.1f} | MAPE: {mape:5.2f}%")
    return {'model': name, 'MAE': mae, 'RMSE': rmse, 'MAPE': mape}

results = []
naive_pred = np.full(TEST_SIZE, train[TARGET].iloc[-1])
results.append(calc_metrics(test[TARGET].values, naive_pred, "Naive (Last Value)"))

src = df_full[TARGET].values
ti = len(df_full) - TEST_SIZE
sn_pred = src[ti - SEASON_LENGTH:ti - SEASON_LENGTH + TEST_SIZE]
results.append(calc_metrics(test[TARGET].values, sn_pred, f"Seasonal Naive ({SEASON_LENGTH})"))

Naive (Last Value)             | MAE:       76.1 | RMSE:      106.4 | MAPE: 12.75%
Seasonal Naive (7)             | MAE:      122.1 | RMSE:      171.6 | MAPE: 17.99%


## LazyPredict Benchmark (Lag-Feature Tabular)

In [12]:
from lazypredict.Supervised import LazyRegressor

ct = len(df_feat) - TEST_SIZE
cv = ct - VAL_SIZE
X_tr = df_feat.iloc[:cv][feature_cols]
y_tr = df_feat.iloc[:cv][TARGET]
X_va = df_feat.iloc[cv:ct][feature_cols]
y_va = df_feat.iloc[cv:ct][TARGET]

reg = LazyRegressor(verbose=0, ignore_warnings=True, custom_metric=None, predictions=True)
lp_models, lp_preds = reg.fit(X_tr, X_va, y_tr, y_va)
print("\nLazyPredict Top 10:")
print(lp_models.head(10).to_string())


LazyPredict Top 10:
                               Adjusted R-Squared  R-Squared        RMSE  Time Taken
Model                                                                               
Lars                                  1050.090612 -79.699278  449.482769    0.026137
KernelRidge                            947.270446 -71.790034  426.888235    0.073908
MLPRegressor                           700.843541 -52.834119  367.119139    0.670765
GaussianProcessRegressor               622.185144 -46.783473  345.873332    0.104008
ExtraTreeRegressor                     160.093109 -11.237931  175.037955    0.013923
AdaBoostRegressor                       37.290199  -1.791554   83.599026    0.200169
DecisionTreeRegressor                   36.711532  -1.747041   82.929833    0.018152
LGBMRegressor                           35.989738  -1.691518   82.087474    0.184617
XGBRegressor                            33.438811  -1.495293   79.038559    1.558969
HistGradientBoostingRegressor           31.3

## FLAML AutoML Run

In [13]:
from flaml import AutoML

automl = AutoML()
automl.fit(
    X_train=X_tr, y_train=y_tr,
    task='regression', time_budget=30, metric='mae',
    estimator_list=['lgbm', 'rf', 'extra_tree', 'catboost'],
    seed=SEED, verbose=0
)
print(f"FLAML best: {automl.best_estimator}")
flaml_val = automl.predict(X_va)
results.append(calc_metrics(y_va.values, flaml_val, f"FLAML ({automl.best_estimator})"))

X_te = df_feat.iloc[ct:][feature_cols]
y_te = df_feat.iloc[ct:][TARGET]
flaml_test = automl.predict(X_te)
results.append(calc_metrics(y_te.values, flaml_test, f"FLAML Test ({automl.best_estimator})"))

FLAML best: lgbm
FLAML (lgbm)                   | MAE:       45.6 | RMSE:       52.4 | MAPE:  9.55%
FLAML Test (lgbm)              | MAE:       69.0 | RMSE:      111.7 | MAPE: 10.23%


## StatsForecast — Statistical Forecasting

In [14]:
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, AutoETS, SeasonalNaive as SFSeasonalNaive

sf_df = df_full[['date', TARGET]].copy()
sf_df.columns = ['ds', 'y']
sf_df['unique_id'] = 'series1'
sf_df = sf_df[['unique_id', 'ds', 'y']]

sf_train = sf_df.iloc[:-TEST_SIZE]
sf = StatsForecast(
    models=[AutoARIMA(season_length=SEASON_LENGTH), AutoETS(season_length=SEASON_LENGTH),
            SFSeasonalNaive(season_length=SEASON_LENGTH)],
    freq=FREQ, n_jobs=1
)
sf.fit(sf_train)
sf_preds = sf.predict(h=TEST_SIZE).reset_index()

y_test_sf = test[TARGET].values
for col in ['AutoARIMA', 'AutoETS', 'SeasonalNaive']:
    if col in sf_preds.columns:
        results.append(calc_metrics(y_test_sf, sf_preds[col].values, f"SF {col}"))
print("StatsForecast complete.")

SF AutoARIMA                   | MAE:      149.6 | RMSE:      178.3 | MAPE: 22.24%
SF AutoETS                     | MAE:      144.5 | RMSE:      173.0 | MAPE: 21.74%
SF SeasonalNaive               | MAE:      148.9 | RMSE:      179.6 | MAPE: 22.22%
StatsForecast complete.


## Top 3 Model Selection

In [15]:
results_df = pd.DataFrame(results).sort_values('MAE').reset_index(drop=True)
print("All Results:")
print(results_df.to_string(index=False))
print("\nTop 3:")
print(results_df.head(3).to_string(index=False))

All Results:
             model        MAE       RMSE      MAPE
      FLAML (lgbm)  45.617569  52.432556  9.547480
 FLAML Test (lgbm)  68.991389 111.681533 10.225801
Naive (Last Value)  76.142857 106.443815 12.749355
Seasonal Naive (7) 122.142857 171.573391 17.991794
        SF AutoETS 144.468430 173.049165 21.737909
  SF SeasonalNaive 148.857147 179.561766 22.223154
      SF AutoARIMA 149.633514 178.295242 22.243772

Top 3:
             model       MAE       RMSE      MAPE
      FLAML (lgbm) 45.617569  52.432556  9.547480
 FLAML Test (lgbm) 68.991389 111.681533 10.225801
Naive (Last Value) 76.142857 106.443815 12.749355


## Final Training and Evaluation of Top 3

In [16]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(test['date'].values, test[TARGET].values, 'ko-', label='Actual', ms=4)
ax.plot(test['date'].values, flaml_test, 's--', label=f'FLAML ({automl.best_estimator})', ms=4)
for col in ['AutoARIMA', 'AutoETS']:
    if col in sf_preds.columns:
        ax.plot(test['date'].values[:len(sf_preds)], sf_preds[col].values, 'o--', label=f'SF {col}', ms=4)
ax.set_title('Forecast vs Actual — Test Set')
ax.legend()
plt.tight_layout()
plt.savefig('forecast_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

## Error Analysis

In [17]:
errors = test[TARGET].values - flaml_test
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(range(len(errors)), errors, alpha=0.7)
axes[0].set_title('Residuals (Actual - FLAML)')
axes[0].axhline(y=0, color='r', linestyle='--')
axes[1].hist(errors, bins=10, edgecolor='black', alpha=0.7)
axes[1].set_title('Residual Distribution')
plt.tight_layout()
plt.savefig('error_analysis.png', dpi=100, bbox_inches='tight')
plt.show()
print(f"Mean residual: {np.mean(errors):.2f}, Std: {np.std(errors):.2f}")

Mean residual: 47.35, Std: 101.15


## Interpretation and Business Insight

- Flight delays peak on Fridays and Mondays (high-traffic days)
- Winter and summer have more delays (storms and thunderstorms)
- Holiday periods amplify delays due to congestion
- Random weather disruptions create unpredictable spikes
- Delay patterns are partly predictable, partly stochastic

## Limitations

1. Synthetic — real delays depend on weather, ATC, mechanical issues
2. No weather forecast features
3. No airport-specific data
4. No distinction between delay causes (weather, mechanical, crew)
5. No cascade effects modeled (one delay causing downstream delays)

## How to Improve This Project

1. Add weather forecast data (ceiling, visibility, wind)
2. Use airport-specific models
3. Model cascade/propagation effects
4. Include ATC ground delay program data
5. Add aircraft utilization and crew schedule features

## Production Considerations

- Hourly delay probability forecasting by airport
- Integration with operations control centers
- Proactive passenger rebooking triggers
- Crew reserve positioning optimization

## Common Mistakes

1. Treating delays as purely random (they have patterns)
2. Not separating weather delays from operational delays
3. Ignoring cascade effects (one delay creates more)
4. Using national averages for airport-specific decisions

## Mini Challenge / Exercises

1. Add a synthetic weather severity index and measure correlation
2. Model winter vs summer delay regimes separately
3. Detect outlier disruption days in the time series
4. Build a binary classifier: will delays exceed a threshold?

## Final Summary / Key Takeaways

1. Flight delays have predictable weekly and seasonal patterns
2. Weather is the dominant cause but hardest to predict long-term
3. Holiday congestion amplifies baseline delay rates
4. Airport-specific and hourly models are needed for operations
5. Proactive delay prediction saves costs and improves passenger experience

In [18]:
import json
metrics = {
    'project': 'Flight Delay Volume Forecasting',
    'best_model': results_df.iloc[0]['model'],
    'best_MAE': float(results_df.iloc[0]['MAE']),
    'best_RMSE': float(results_df.iloc[0]['RMSE']),
    'best_MAPE': float(results_df.iloc[0]['MAPE']),
    'all_results': results_df.to_dict('records')
}
with open('metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print("Metrics saved.")
print("\n=== Flight Delay Volume Forecasting — Complete ===")

Metrics saved.

=== Flight Delay Volume Forecasting — Complete ===
